### Steps 1, 2 & 3: Corpus, Taxonomy, Knowledge Base & Preprocessing Imports

In [1]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

corpus = [
    ("I have a sharp pain in my chest", "emergency"),
    ("My chest hurts badly and I cannot breathe", "emergency"),
    ("I am having severe difficulty breathing", "emergency"),
    ("Sudden numbness on one side of my body", "emergency"),
    ("He collapsed and is unresponsive", "emergency"),
    ("I need to book an appointment with a doctor", "booking"),
    ("Can I schedule a visit for tomorrow morning", "booking"),
    ("How do I book a consultation with a cardiologist", "booking"),
    ("I want to change my appointment time", "booking"),
    ("Cancel my appointment for Friday", "booking"),
    ("What are the side effects of Paracetamol", "medication"),
    ("How often should I take Ibuprofen", "medication"),
    ("Are there any side effects for Amoxicillin", "medication"),
    ("What is the dosage for Aspirin", "medication"),
    ("Can I take multivitamins daily", "medication"),
    ("Where is the nearest pharmacy", "location"),
    ("How do I find a clinic near me", "location"),
    ("Is there an open hospital in this area", "location"),
    ("Give me the address of City Health Clinic", "location"),
    ("Find emergency rooms nearby", "location"),
    ("How can I manage high blood pressure", "advice"),
    ("What should I eat to lower my cholesterol", "advice"),
    ("Tips for maintaining healthy blood sugar", "advice"),
    ("How to treat a minor burn at home", "advice"),
    ("What are some healthy habits for heart health", "advice"),
    ("I feel slightly tired today", "general"),
    ("Hello medical assistant", "general"),
    ("How is the weather today", "general"),
    ("What is your name", "general"),
    ("Can you help me with something", "general"),
    ("My head is hurting slightly", "advice"),
    ("Suggest remedies for a common cold", "advice"),
    ("Can you find me a doctor", "booking"),
    ("Is there a 24 hour pharmacy open", "location"),
    ("Are antibiotics safe during pregnancy", "medication"),
    ("I am experiencing severe chest pressure", "emergency"),
    ("Please arrange a doctor visit", "booking"),
    ("Is Ibuprofen safe for kids", "medication"),
    ("Where is the local general hospital", "location"),
    ("Help me with daily exercise tips", "advice")
]

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


### Step 4: NLP Preprocessing Pipeline

In [2]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    filtered_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(filtered_tokens)

print("Example preprocessing:")
print(preprocess_text("I have a sharp pain in my chest!"))

Example preprocessing:
sharp pain chest


### Step 5: Feature Extraction & Classifier Training

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import numpy as np

texts, labels = zip(*corpus)
cleaned_texts = [preprocess_text(t) for t in texts]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(cleaned_texts)
y = np.array(labels)

model = LogisticRegression(C=1.0, class_weight='balanced')
model.fit(X, y)
print("Model trained successfully.")

Model trained successfully.


### Step 6: Evaluation Metrics

In [4]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X)
print(classification_report(y, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y, y_pred))

              precision    recall  f1-score   support

      advice       1.00      1.00      1.00         8
     booking       1.00      1.00      1.00         7
   emergency       1.00      1.00      1.00         6
     general       1.00      1.00      1.00         5
    location       1.00      1.00      1.00         7
  medication       1.00      1.00      1.00         7

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
[[8 0 0 0 0 0]
 [0 7 0 0 0 0]
 [0 0 6 0 0 0]
 [0 0 0 5 0 0]
 [0 0 0 0 7 0]
 [0 0 0 0 0 7]]


### Steps 7 & 8: Knowledge Base & Decision Rules

In [5]:
knowledge_base = {
    "booking": "You can register your details and pick available slots via our medical booking system at booking.example.com.",
    "medication": "Please refer to the pharmacist's instructions or consult with a qualified doctor before altering any dosage.",
    "location": "The nearest medical centers and pharmacies are listed inside our directory at local-health.example.com.",
    "advice": "For non-urgent issues, maintain hydration, rest, and monitor your symptoms closely.",
    "general": "I am your digital assistant. Let me know if you need info on bookings, medication, locations, or health advice."
}

def analyze_query(query):
    clean_q = preprocess_text(query)

    if any(word in clean_q for word in ['chest', 'breathing', 'collapse', 'unresponsive', 'stroke', 'emergency', 'dying']):
        return "EMERGENCY", "CRITICAL: Please call emergency services (911) immediately or proceed to the nearest emergency room."

    if not clean_q or all(word in stop_words for word in query.split()):
        return "UNSUPPORTED", "I'm sorry, I cannot recognize health-related topics in this query. Please ask about clinics, bookings, or medications."

    vec_q = vectorizer.transform([clean_q])
    probs = model.predict_proba(vec_q)[0]
    max_idx = np.argmax(probs)
    confidence = probs[max_idx]
    intent = model.classes_[max_idx]

    if confidence < 0.28:
        return "LOW_CONFIDENCE", "I'm not fully sure of your request. Did you mean to ask about: " + ", ".join(model.classes_) + "?"

    return intent, knowledge_base.get(intent, "No detailed records found.")

### Steps 9 & 10: Run the Interactive Assistant Tests

In [6]:
test_queries = [
    "I need to schedule a consultation with a doctor",
    "Suggest remedies for a common cold",
    "How do I get to the local clinic",
    "What are the side effects of Amoxicillin",
    "Is there a hospital close to my home",
    "Please set up an appointment for next Monday",
    "Can you write a poem about flowers?",
    "Uhm ok cool",
    "Help, my chest hurts and I can't breathe"
]

for q in test_queries:
    intent, response = analyze_query(q)
    print(f"Query: {q}")
    print(f"Detected Intent: {intent}")
    print(f"Response: {response}")
    print("-" * 50)

Query: I need to schedule a consultation with a doctor
Detected Intent: booking
Response: You can register your details and pick available slots via our medical booking system at booking.example.com.
--------------------------------------------------
Query: Suggest remedies for a common cold
Detected Intent: advice
Response: For non-urgent issues, maintain hydration, rest, and monitor your symptoms closely.
--------------------------------------------------
Query: How do I get to the local clinic
Detected Intent: location
Response: The nearest medical centers and pharmacies are listed inside our directory at local-health.example.com.
--------------------------------------------------
Query: What are the side effects of Amoxicillin
Detected Intent: medication
Response: Please refer to the pharmacist's instructions or consult with a qualified doctor before altering any dosage.
--------------------------------------------------
Query: Is there a hospital close to my home
Detected Intent: 